# Lamer Konekte — Gemma 4 multimodal QLoRA (Training Mode A)
E2B-first per the resource plan; the hardware gate below runs before anything else.
Acceptance criteria: docs/TRAINING_PLAN.md §18E — the adapter integrates only if held-out
evaluation beats the hosted baseline on ≥1 target metric with no safety regression.

In [ ]:
# KAGGLE HARDWARE GATE - runs before any training (docs/TRAINING_PLAN.md 18C)
import shutil, subprocess, torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    print("BF16 support:", torch.cuda.is_bf16_supported())
    print("CUDA:", torch.version.cuda)
print("free disk GB:", round(shutil.disk_usage("/kaggle/working").free / 1e9, 1))
assert torch.cuda.is_available(), "GPU required - select a GPU accelerator in Kaggle settings"


In [ ]:
# TRAINING MODE A - multimodal QLoRA on Gemma 4 E2B (E4B only after a successful memory test)
# Requires: HF token in Kaggle Secrets ("HF_TOKEN") + accepted Gemma licence on Hugging Face.
%pip install -q -U transformers peft bitsandbytes accelerate datasets

import json, os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

MODEL_ID = "google/gemma-4-e2b-it"   # E2B first per the resource plan
DATA = "/kaggle/input/lamer-konekte-training/"  # private dataset uploaded by scripts/kaggle_push_training.sh

import torch
from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
                                         task_type="CAUSAL_LM",
                                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
model.print_trainable_parameters()


In [ ]:
# Warm-up run first (32 records, 10 steps) - abort on OOM before committing to the full run.
# Full run: batch 1 + gradient accumulation 8, gradient checkpointing, short image resolution.
import json
from datasets import Dataset

def load_split(name):
    rows = [json.loads(l) for l in open(DATA + f"{name}.jsonl")]
    return Dataset.from_list([r for r in rows if r["record_type"] != "image_identification" or r["image_path"]])

train_ds, val_ds = load_split("train"), load_split("validation")
print("train:", len(train_ds), "validation:", len(val_ds))

from transformers import TrainingArguments, Trainer

def collate(batch):
    texts = [f"<start_of_turn>user\n{r['instruction']}\nCandidates: {r.get('candidate_species')}<end_of_turn>\n"
             f"<start_of_turn>model\n{json.dumps(r.get('expected_json') or {'safe': r.get('expected_safe_response')}, ensure_ascii=False)}<end_of_turn>"
             for r in batch]
    enc = processor.tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
    enc["labels"] = enc["input_ids"].clone()
    return enc

args = TrainingArguments(output_dir="/kaggle/working/adapter", per_device_train_batch_size=1,
                         gradient_accumulation_steps=8, gradient_checkpointing=True, bf16=True,
                         num_train_epochs=2, logging_steps=5, save_strategy="epoch",
                         eval_strategy="epoch", report_to="none", max_steps=-1)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=collate)
trainer.train()
trainer.save_model("/kaggle/working/adapter/final")
print("adapter saved to /kaggle/working/adapter/final")
